# TM1py: Writing Cells

This module is the fifth chunk of the tm1py course. Readers are
assumed to have absorbed the earlier chunks: the mental model from
chunk 1, the connect-and-list pattern from chunk 2, the cube and
dimension navigation from chunk 3, and the cell read methods from
chunk 4. The audience remains TM1 expert. TI's `CellPutN`,
sandboxes, transaction logs, and feeders are taken for granted; this
chunk covers how their tm1py equivalents are shaped.

The goal is to push values back into a TM1 cube and to choose the
right method for the situation. Where reading was three methods that
each fit a clear use case, writing is harder. tm1py exposes at least
five write entry points on `tm1.cells`, and the choice between them
is consequential. Volume, transactional behaviour, parallelism, and
whether the target cube carries rules all change which method is
correct.

The chunk avoids dumping the menu up front. It starts with a single
method, `write_dataframe`, that mirrors the read pattern from chunk 4
and works for the spreadsheet-like cases that account for most of
practical use. It then introduces the alternatives in the order the
questions naturally arise: "what about a single cell?", "what about
adding to existing values rather than replacing?", "what about
millions of cells?", "what about staging the write before committing
it?", "what about parallel writes against a busy cube?". Each method
is shown only after the situation that motivates it.

Writing without sandboxes is dangerous in any shared TM1 model.
Writing with sandboxes is the professional practice. A topic of its
own covers the sandbox-based workflow because it is the path that
turns "experimental Python script" into "safe-to-run production
code."

The topics below are arranged linearly for review. The Sales Plan
cube continues as the running example. The same tidy DataFrame shape
returned by `execute_view_dataframe` in chunk 4 is the input to most
of the writes shown here, which is the read–transform–write pattern
that is the whole reason the library exists.

---

## Topic list

1. The writing scenario
2. write_dataframe: the starting point
3. Single cells with write_value
4. The dict form: write
5. Replace vs increment
6. Sandboxes: staging writes safely
7. Volume: blob writes and unbound processes
8. Parallelism with write_async
9. Rules and feeders during writes
10. Choosing the right write method
11. Real-world design principles
12. Common mistakes

---

## 1. The writing scenario

Reading and writing are not symmetric. A read of a TM1 cube is a
question: it returns data, it does not change anything, it is safe to
retry, and the cost of getting it wrong is at most a wrong answer in
Python. A write modifies the system of record. A wrong write can
overwrite a colleague's plan, break a downstream report, trigger
unwanted feeder propagation, or fill a transaction log to the point
where it has to be archived manually. Writes deserve more thought
than reads, and the library acknowledges this by offering several
distinct write methods rather than one.

Writes also have to negotiate concurrency, volume, and rule effects
in a way that reads do not. A read pulls a snapshot through HTTP and
parses it on the client; a write submits a payload and waits for the
server to apply it, possibly recalculating consolidations, possibly
firing feeders, possibly contending with another writer for a lock on
the same cube. The right write method depends on which of those
concerns dominates.

The starting question for any write is "how many cells, into which
cube, and with what guarantees?" Answer that, and the choice of
method follows. The rest of the chunk works through the answers, in
the order the questions usually appear.

The running example is a 2026 plan derived from 2025 actuals: read
the actuals, scale by an assumed growth, write the result back as
the plan version. The shape of the data is the tidy DataFrame from
chunk 4, with columns Period, Region, Product, Version, Measure, and
Value.

## 2. write_dataframe: the starting point

The most pandas-like write method is `write_dataframe`. It takes a
cube name and a tidy DataFrame whose columns name the dimensions and
whose final column carries the value. The library matches columns to
dimensions in cube order and submits the cells in one HTTP request.

In [ ]:
import pandas as pd
from TM1py import TM1Service

plan = pd.DataFrame({
    "Period":  ["2026Q1", "2026Q1", "2026Q1"],
    "Region":  ["Europe", "Americas", "Asia"],
    "Product": ["Standard", "Standard", "Standard"],
    "Version": ["Plan", "Plan", "Plan"],
    "Measure": ["Revenue", "Revenue", "Revenue"],
    "Value":   [126_000.0, 262_500.0, 94_500.0],
})

with TM1Service(**creds) as tm1:
    tm1.cells.write_dataframe(
        cube_name="Sales Plan",
        data=plan,
    )

The DataFrame is the inverse of what `execute_view_dataframe`
returned in chunk 4: one column per dimension in cube order, plus a
single value column at the end. Because the read produces this shape
naturally and the write consumes it directly, an entire
read-transform-write pipeline can pass a single DataFrame through
without reshaping.

In [ ]:
with TM1Service(**creds) as tm1:
    actuals = tm1.cells.execute_view_dataframe(
        cube_name="Sales Plan", view_name="2025 Actuals"
    )
    plan = (
        actuals
        .query("Measure == 'Revenue'")
        .assign(
            Period=lambda d: d["Period"].str.replace("2025", "2026"),
            Version="Plan",
            Value=lambda d: (d["Value"] * 1.05).round(2),
        )
    )
    tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)

`write_dataframe` defaults to replace semantics: the value at each
address overwrites whatever was there. It defaults to the base values
of the cube, not a sandbox. It defaults to one HTTP request,
synchronous, blocking until the server has accepted the payload.
These defaults are correct for most situations and the right starting
point. The remaining topics in this chunk are about the situations
where one of the defaults needs to change.

The dimension column order matters. By default tm1py assumes the
columns appear in cube dimension order, with the value as the last
column. To use a DataFrame whose columns are in a different order or
whose names differ from the dimension names, pass a `dimensions`
argument that names the dimension columns in cube order. Reaching
for `cube.dimensions` from chunk 3 to build that list is the
disciplined approach.

## 3. Single cells with write_value

Some writes are a single cell at a known address. The method is
`write_value`, the inverse of `get_value` from chunk 4, and the
shape mirrors TI's `CellPutN`.

In [ ]:
with TM1Service(**creds) as tm1:
    tm1.cells.write_value(
        value=125_000.0,
        cube_name="Sales Plan",
        element_tuple=("2026Q1", "Europe", "Standard", "Plan", "Revenue"),
    )

The `element_tuple` is the cell address as a Python tuple, in cube
dimension order. The same caution that applied to `get_value` in
chunk 4 applies here: the order is on the cube, not in the script,
and constructing the tuple from `cube.dimensions` is more robust than
hardcoding it.

`write_value` is the right method for a small number of point
updates: a single corrected cell, a parameter written to a control
cube, an isolated overwrite. It is the wrong method for any loop
over more than a handful of cells, where the per-call HTTP cost
dominates exactly as it did for `get_value`. Anything that touches
many cells at once should be expressed as a DataFrame or a dict and
written in one batched call.

For string measures, the value passed in is a `str` rather than a
`float`. The cube's measure dimension determines which form is
expected; passing a number to a string measure or a string to a
numeric measure raises a clear server-side error.

## 4. The dict form: write

`write_dataframe` is convenient when the data is already a tidy
DataFrame. When the data is naturally a dictionary, perhaps because
it came from `execute_view` or `execute_mdx` (which return cellsets
as dicts keyed by element tuple), `write` is the more direct entry
point.

In [ ]:
updates = {
    ("2026Q1", "Europe",   "Standard", "Plan", "Revenue"): 125_000.0,
    ("2026Q1", "Americas", "Standard", "Plan", "Revenue"): 260_000.0,
    ("2026Q1", "Asia",     "Standard", "Plan", "Revenue"):  94_500.0,
}

with TM1Service(**creds) as tm1:
    tm1.cells.write(
        cube_name="Sales Plan",
        cellset_as_dict=updates,
    )

The dict keys are element tuples in cube order; the values are the
values to write. The call submits the entire dict as one HTTP
request, the same way `write_dataframe` does, and accepts the same
optional flags for sandbox, increment, and so on.

`write` and `write_dataframe` are two surfaces over the same
underlying operation. The DataFrame form is more pandas-like and is
the right entry point when data is already tabular; the dict form is
more direct when data is already keyed. Reaching from one to the
other is a one-line conversion in either direction:
`df.set_index([...]).to_dict()["Value"]` produces the dict form, and
`pd.DataFrame(updates.items(), columns=[...])` produces the
DataFrame form. The choice between them is convenience at the call
site, not capability.

## 5. Replace vs increment

By default, every write replaces the value at its address. The new
value overwrites whatever was there before. This is correct for plans
that are written wholesale and for any situation where the source
data already represents the desired final state.

For situations where the new value should be added to the existing
value rather than replace it, the keyword is `increment=True`. The
canonical case is appending journal entries to a fact-style cube, or
posting deltas from an upstream system that emits changes rather
than full snapshots.

In [ ]:
journal = pd.DataFrame({
    "Period":  ["2026Q1", "2026Q1"],
    "Account": ["4000",  "4000"],
    "Region":  ["Europe", "Americas"],
    "Version": ["Actual", "Actual"],
    "Measure": ["Amount", "Amount"],
    "Value":   [50_000.0, 75_000.0],
})

with TM1Service(**creds) as tm1:
    tm1.cells.write_dataframe(
        cube_name="General Ledger",
        data=journal,
        increment=True,
    )

Increment writes are not idempotent. A replace write of the same
data twice produces the same end state; an increment write of the
same data twice doubles the values. Scripts that retry on failure
must distinguish between replace semantics (safe to retry) and
increment semantics (must not retry without compensating, or must
record what was written and check before retrying).

The right default for most planning workflows is replace. Increment
is a deliberate choice for fact ingestion, and the script that uses
it should be obvious about why.

## 6. Sandboxes: staging writes safely

A write that goes straight to the base values of a cube is committed
the moment the call returns. There is no preview, no approval step,
no automatic backup. If the values were wrong, the recovery path
runs through the transaction log and is at best inconvenient. For a
shared production cube this risk is rarely worth taking.

Sandboxes are TM1's mechanism for staging writes without touching the
base. A sandbox is a per-user shadow of the cube; values written into
a sandbox are visible to that user but invisible to the base values
and to other users until the sandbox is merged. The same mechanism
underlies PAW's "personal workspace" feature.

Every tm1py write method accepts a `sandbox_name` argument that
directs the write into a specific sandbox.

In [ ]:
with TM1Service(**creds) as tm1:
    tm1.cells.write_dataframe(
        cube_name="Sales Plan",
        data=plan,
        sandbox_name="2026_Plan_Draft",
    )

Listing, creating, and committing sandboxes are operations on
`tm1.sandboxes`, the SandboxService.

In [ ]:
from TM1py import Sandbox

with TM1Service(**creds) as tm1:
    if not tm1.sandboxes.exists("2026_Plan_Draft"):
        tm1.sandboxes.create(Sandbox(name="2026_Plan_Draft"))

    tm1.cells.write_dataframe(
        cube_name="Sales Plan",
        data=plan,
        sandbox_name="2026_Plan_Draft",
    )

    # later, after review:
    tm1.sandboxes.merge(sandbox_name="2026_Plan_Draft")

The two-step pattern, write to a sandbox, review, merge, is the
professional default for any non trivial write. Costs are modest:
sandboxes are cheap on the server, and the merge step is one extra
call. Benefits are large: a wrong sandbox is discarded with
`tm1.sandboxes.delete`, a right sandbox is committed atomically,
and at no point during development does the base cube show
intermediate state.

For scripts that genuinely should write straight to base values, the
absence of a `sandbox_name` is the way to do it. For everything
else, default to a sandbox and treat going direct to base as the
deliberate exception.

## 7. Volume: blob writes and unbound processes

For small writes, an HTTP request carrying a JSON payload is fine.
For very large writes, hundreds of thousands or millions of cells,
the JSON-over-HTTP path becomes the bottleneck: the payload is
verbose, the parsing is costly on both sides, and the server is
forced to apply each cell through the REST endpoint rather than
through TM1's faster server-side write paths.

tm1py offers two faster routes for high volume writes, both
accessible through keyword arguments on the same `write_dataframe`
and `write` methods covered above.

`use_blob=True` uploads the payload as a blob and triggers a
server-side process to consume it. The wire format is binary rather
than JSON, the server-side write goes through TM1's bulk write path,
and the speed improvement is typically several multiples for large
writes.

In [ ]:
tm1.cells.write_dataframe(
    cube_name="Sales Plan",
    data=very_large_plan,
    use_blob=True,
)

`use_ti=True` (also exposed as `write_through_unbound_process`) uses
an unbound TI process to perform the write server-side. The values
are sent to the server, and the server runs a TI that issues
`CellPutN` calls in process with the cube. This is the fastest path
available for very large writes and the closest equivalent to writing
the same data through a regular TI process invoked from PAW.

In [ ]:
tm1.cells.write_dataframe(
    cube_name="Sales Plan",
    data=very_large_plan,
    use_ti=True,
)

The two are not interchangeable. Blob writes are the safer general
purpose default for large volumes; the unbound process path is the
fastest but pays for that speed by leaving any error handling to the
TI script that runs server-side, and the error reporting is
correspondingly coarser. For most workflows, `use_blob=True` is the
right setting once volume crosses into the tens of thousands of
cells; reach for `use_ti=True` only when blob writes are still too
slow and the TI cost is acceptable.

There is no hard threshold at which one method becomes correct.
Measure the actual write time at the actual volume against the
actual cube. The right answer depends on the cube's rules, the
server's load, the network, and the size of the payload, and the
only reliable way to choose is to time both options against
representative data.

## 8. Parallelism with write_async

Even with blob or unbound process writes, very large jobs can take
minutes. Splitting the payload into pieces and posting them in
parallel can shorten the wall clock time, at the cost of more
concurrent connections to the server.

`write_async` does this automatically. It takes a dict of cells and
internally splits the work into slices, posting each slice on its
own connection, and reassembles the result.

In [ ]:
with TM1Service(**creds) as tm1:
    tm1.cells.write_async(
        cube_name="Sales Plan",
        cells=very_large_dict,
        slice_size=250_000,
        max_workers=8,
    )

`slice_size` is the number of cells per request; `max_workers` is
the number of concurrent connections. Both have sensible defaults,
and the right values depend on the server's capacity. A server under
heavy load by other users may not benefit from eight concurrent
write connections; a quiet server may handle sixteen. As with the
volume-vs-speed tradeoff in topic 7, the only reliable way to tune
is to time the actual job against the actual server.

`write_async` is the right tool when the write is large enough that
splitting it shortens the wall clock time and the server has the
headroom to accept the additional connections. It is the wrong tool
when those conditions do not hold, because the additional
parallelism then costs server resources without delivering speed.
For most planning workflows, a single `write_dataframe` with
`use_blob=True` is fast enough and simpler. Reach for `write_async`
only when timed measurement says it helps.

## 9. Rules and feeders during writes

Writing into a cube that has rules is mechanically the same as
writing into one that does not, but the consequences differ. A rule
that depends on the cells being written may recompute, possibly
across the cube hierarchy and possibly into other cubes. Feeders
that the rule declares determine which downstream cells need to be
recalculated and may extend the write's effect well beyond the cells
named in the payload.

The library does not have a special "rules-aware write" method
because the rule application happens server-side and is determined
by the cube's rule definition, not by the API. Two consequences
follow.

First, write performance can be dominated by rule and feeder
recalculation rather than by the write itself. A small write into a
heavily ruled cube can take longer than a large write into a cube
without rules. Profiling time is the only reliable way to know.

Second, partial writes can leave the cube in a state that is
internally consistent only if every dependent cell has been written.
Writing only the leaves of a consolidated tree is fine when the cube
recomputes consolidations automatically; writing only some of the
fact cells that feed a rule may leave dependent cells stale until
the remaining writes complete. Sandboxes mitigate this by deferring
the visibility of any change until merge time, and the sandbox-based
workflow from topic 6 is even more valuable for writes into ruled
cubes than for writes into unruled ones.

For situations where the write should not trigger feeders, the
keyword `deactivate_transaction_log=True` on `write` and
`write_dataframe` skips logging (and therefore breaks
recoverability). This is appropriate only for very specific
high-volume bulk loads, in maintenance windows, where the cube is
being reset rather than incrementally updated. The corresponding
`reactivate_transaction_log=True` reenables logging at the end of
the write. Treat these flags as expert options rather than tuning
knobs; the default of "log everything" is right for almost every
real workflow.

## 10. Choosing the right write method

The decisions stack roughly in this order.

How many cells are being written? For one to a handful, `write_value`.
For dozens to thousands, `write_dataframe` (or `write`) without
extra flags. For tens of thousands or more, `write_dataframe` with
`use_blob=True`. For millions, `write_dataframe` with `use_ti=True`,
or `write_async` to split a blob write across connections.

Where should the write land? In a sandbox during development,
testing, scenario work, or any non trivial change to a shared cube.
Direct to base only when the write is genuinely meant to be the new
canonical state and has been reviewed.

Should the new value replace or be added to the existing value? For
planning, replace (the default). For fact ingestion or journal
posting, increment.

Does the cube have rules that the write will trigger? If yes,
profile and consider whether the write should be done in pieces with
sandboxes between them, or whether a TI process running server-side
would be faster.

Does the script need to be retry safe? Replace writes are idempotent
and safe to retry. Increment writes are not. Either change to replace
semantics, or implement deduplication so that a retry does not
double-post.

Three of these decisions, the volume one, the sandbox one, and the
replace versus increment one, are the ones that matter for almost
every script. The remaining two come up only for cubes with
significant rule logic or for production scripts that need
guaranteed-once delivery.

## 11. Real-world design principles

**Default to write_dataframe.** It covers the spreadsheet-shaped
case that accounts for most writes, mirrors the read shape from
chunk 4, and pairs naturally with pandas transformations. Reach for
the alternatives only when the situation specifically demands them.

**Default to a sandbox.** For any write into a shared cube during
development, scenario work, or non trivial production changes, the
write should land in a sandbox first. Merging a sandbox after review
is one extra call. Recovering a wrong write from a transaction log
is many calls and an unhappy afternoon.

**Match volume to method.** Small writes are fine over plain JSON.
Large writes pay for the JSON overhead in seconds or minutes;
`use_blob=True` is the standard fix once the payload crosses the
tens of thousands of cells mark. Reaching for `use_ti=True` or
`write_async` is a tuning step after measurement, not before.

**Prefer replace over increment.** Replace writes are idempotent and
forgive retries; increment writes do not. When the upstream system
emits deltas rather than snapshots, prefer to accumulate them in
Python or in a staging cube and write the final snapshot, rather
than streaming deltas as increment writes against the production
cube.

**Read dimension order from the cube.** As in chunk 4, the cube is
authoritative for dimension order. When the DataFrame's column
order matches `cube.dimensions`, no `dimensions` argument is needed;
when it does not, the argument should be built from `cube.dimensions`
rather than typed by hand.

**Profile before optimizing.** The choice between blob writes,
unbound process writes, and async parallelism is not a function of
data size alone. The same one-million-cell write can be fastest
through any of the three depending on the cube's rules, the
server's load, and the network. Time the actual job before
committing to a particular flag.

## 12. Common mistakes

A short collection of errors that come up while learning to write
cells through tm1py.

**Looping `write_value` over many cells.** The mirror image of the
`get_value` looping mistake from chunk 4. Each call is one HTTP
round trip; thousands of calls are thousands of round trips. The
remedy is a single `write_dataframe` or `write`.

In [ ]:
# Wrong: one round trip per cell
for region in regions:
    for product in products:
        tm1.cells.write_value(
            value=values[region, product],
            cube_name="Sales Plan",
            element_tuple=("2026Q1", region, product, "Plan", "Revenue"),
        )

# Correct: one round trip total
tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan_df)

**Writing straight to base when a sandbox would do.** A development
script that writes to base is one bug away from corrupting a shared
cube. The remedy is a sandbox.

In [ ]:
# Wrong: experimental write hits the base values
tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)

# Correct: stage in a sandbox, review, merge
tm1.cells.write_dataframe(
    cube_name="Sales Plan", data=plan,
    sandbox_name="2026_Plan_Draft",
)

**Confusing replace and increment semantics.** Running an increment
write twice doubles the values; running a replace write twice is a
no-op. The bug surfaces as "the numbers came out twice as large as
expected after the script was rerun."

In [ ]:
# Wrong: increment used where replace was meant
tm1.cells.write_dataframe(
    cube_name="Sales Plan", data=plan, increment=True,
)
# rerunning the script doubles the plan

# Correct: replace is the default and the right choice for a plan
tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)

**Hardcoding dimension column order in the DataFrame.** A column
order that does not match the cube produces a write that lands in
the wrong cells, with no error: the values are valid floats, the
cells exist, only the mapping is wrong.

In [ ]:
# Wrong: column order does not match cube.dimensions
plan = pd.DataFrame({
    "Region":  [...], "Period":  [...], "Product": [...],
    "Version": [...], "Measure": [...], "Value":   [...],
})
tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)

# Correct: reorder to match the cube
cube = tm1.cubes.get("Sales Plan")
plan = plan[cube.dimensions + ["Value"]]
tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)

**Omitting `use_blob=True` for large writes.** A million-cell write
through plain JSON can take an order of magnitude longer than the
same write through a blob upload. The fix is one keyword.

In [ ]:
# Wrong: slow for large volumes
tm1.cells.write_dataframe(cube_name="Sales Plan", data=large_plan)

# Correct: blob path for high volumes
tm1.cells.write_dataframe(
    cube_name="Sales Plan", data=large_plan, use_blob=True,
)

**Disabling the transaction log without a recovery plan.** The
`deactivate_transaction_log=True` flag is a sharp tool for one
specific kind of bulk reset. Used casually, it removes the only
record of what changed, and a wrong write is unrecoverable.

In [ ]:
# Wrong: routine write with the safety net turned off
tm1.cells.write_dataframe(
    cube_name="Sales Plan", data=plan,
    deactivate_transaction_log=True,
)

# Correct: leave the default on; the cost is small, the safety is large
tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)

**Retrying an increment write on a generic exception.** A retry of a
replace write is safe; a retry of an increment write doubles the
data. A try-except that retries blindly without distinguishing the
two will silently inflate values.

In [ ]:
# Wrong: blind retry on any exception
for attempt in range(3):
    try:
        tm1.cells.write_dataframe(
            cube_name="GL", data=journal, increment=True,
        )
        break
    except TM1pyException:
        continue
# if the first attempt half-succeeded server-side, the retry doubles
# everything that did get through

# Correct: idempotent shape, or explicit deduplication, or no retry
tm1.cells.write_dataframe(cube_name="GL", data=journal_snapshot)
# replace semantics, retry-safe

**Forgetting to merge a sandbox.** A sandbox-based write that nobody
ever merges is a write that never reaches the base cube. The plan
looks right in the script's user's PAW workspace and is invisible to
everyone else.

In [ ]:
# Wrong: written to sandbox, never merged
tm1.cells.write_dataframe(
    cube_name="Sales Plan", data=plan,
    sandbox_name="2026_Plan_Draft",
)
# end of script; sandbox sits there indefinitely

# Correct: merge after review (interactive) or as the next step
tm1.cells.write_dataframe(
    cube_name="Sales Plan", data=plan,
    sandbox_name="2026_Plan_Draft",
)
# ... review ...
tm1.sandboxes.merge(sandbox_name="2026_Plan_Draft")